# Semantic Routing for RAG

An implementation of **semantic routing** — selecting which prompt (or downstream chain) to use
for a question based on **embedding similarity**, rather than an LLM classification call.

**Why it helps:** [logical routing](#) asks an LLM to classify each question into a category,
which costs an extra LLM call and can be inconsistent on ambiguous phrasing. Semantic routing
instead embeds a small set of *candidate prompts* once, up front, and at query time just embeds
the incoming question and picks whichever candidate prompt is closest by cosine similarity. This
is cheaper (no extra LLM call to route) and deterministic (same embedding model, same distance
metric every time).

**Pipeline:**
1. Define a small set of candidate prompts, each specialized for a domain (physics, math).
2. Embed every candidate prompt once, up front.
3. At query time, embed the incoming question and compute its cosine similarity to each candidate.
4. Pick the prompt with the highest similarity and use it to answer the question.

**Stack:** LangChain, Ollama (embedding model + local chat LLM).

> **Fixes from the original draft:** `prompt_router` was reading a module-level `query` variable
> instead of the question actually passed into the chain (`input["query"]`) -- meaning every
> invocation would route based on whatever `query` happened to be set to elsewhere, not the
> current question. It's now read from the chain's input as intended. A `num_cxt` → `num_ctx`
> typo in the `ChatOllama` call was also fixed, and `query` is defined explicitly before use.


## Setup

Load model names from environment variables (`OLLAMA_MODEL` for the chat model, `OLLAMA_EMBED_LITE` for the embedding model) via a `.env` file.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL")
EMBED_MODEL = os.getenv("OLLAMA_EMBED_LITE")

## 1. Define Candidate Prompts

Each template is a full system-style prompt specialized for one domain. The router will choose between these based on which one the question is semantically closest to.

In [2]:
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings

# Two candidate prompts
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""


C:\Users\chett\AppData\Local\Temp\ipykernel_26260\700526818.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utils.math import cosine_similarity
c:\Users\chett\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


## 2. Embed the Candidate Prompts

Embed each candidate prompt once, up front -- this only needs to happen a single time, not on every query, since the candidate set doesn't change.

In [3]:
# Embed each candidate prompt once, up front
embeddings = OllamaEmbeddings(model=EMBED_MODEL)
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)


## 3. Build the Router

At query time: embed the incoming question, compute its cosine similarity against every candidate prompt embedding, and pick the prompt with the highest similarity (`similarity.argmax()` gives the *index* of the best match, which is used to look up the corresponding template).

In [4]:
# Route a question to whichever candidate prompt it's most semantically similar to
def prompt_router(input):
    # Embed the incoming question (NOT the module-level `query` -- use the chain's input)
    query_embedding = embeddings.embed_query(input["query"])
    # Compare against each candidate prompt's embedding
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    most_similar = prompt_templates[similarity.argmax()]
    # Log which prompt was chosen
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)


## 4. Assemble & Run the Chain

`RunnablePassthrough()` forwards the raw question into the chain under the `"query"` key, `prompt_router` picks and fills in the right prompt template, and the result flows into the LLM.

In [5]:
chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | ChatOllama(model=OLLAMA_MODEL, num_ctx=1024, temperature=0)
    | StrOutputParser()
)

query = "What's a black hole?"
print(chain.invoke(query))


Using PHYSICS
A black hole is a region in space where the gravitational pull is so strong that nothing, including light, can escape. It's formed when a massive star collapses in on itself and its gravity becomes so strong that it warps the fabric of spacetime around it.

Imagine spacetime as a trampoline. If you place a heavy object, like a bowling ball, on the trampoline, it will warp and curve, creating a depression. A black hole is like that depression, but instead of being a temporary effect, it's a permanent feature of spacetime.

The point of no return, called the event horizon, marks the boundary of a black hole. Once you cross the event horizon, you're trapped by the black hole's gravity and can't escape. That's why black holes are so fascinating and mysterious – they're like cosmic vacuum cleaners that suck in everything that gets too close.

Does that make sense?


## Next Steps

- Add more candidate prompts (e.g. biology, history) to route across a broader set of domains.
- Combine with [logical routing](#): use semantic routing for cheap first-pass triage, falling back to an LLM classification call only when the top two similarity scores are close.
- Cache `prompt_embeddings` to disk so they aren't recomputed on every notebook restart.